# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [17]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except SyntaxError as e:
        return f"Invalid mathematical expression: {e}"
    except Exception as e:
        return f"Error in calculation: {e}"

In [18]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

In [19]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [20]:
# 🛠️ TOOL 2: Keyword Extractor (with stop word filtering)

def extract_keywords(text: str) -> list:
    """Extract keywords from text, filtering out stop words."""
    try:
        words = text.split()
        # Filter out stop words and words with length <= 4 as before
        filtered_words = [w.lower() for w in words if len(w) > 4 and w.lower() not in stop_words]
        keywords = list(set(filtered_words))
        return keywords[:5]
    except Exception as e:
        return []

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

### 🚀 Router Function

To make the agent logic more modular and explicit, I'll create a dedicated `router_function`. This function will analyze the input query and determine the appropriate `action` (e.g., 'calculate', 'keywords', 'general') and any relevant `payload` needed for that action.

The `agent` function will then call this router to get the intent and execute the corresponding tool or response.

In [21]:
def router_function(query: str) -> dict:
    """Routes the query to the appropriate action based on keywords."""
    query_lower = query.lower()

    if "calculate" in query_lower:
        expression = query_lower.replace("calculate", "").strip()
        return {"action": "calculate", "payload": expression}
    elif "keywords from" in query_lower: # Look for 'keywords from' to isolate the text
        # Extract the text after "keywords from"
        text_for_keywords = query[query_lower.find("keywords from") + len("keywords from"):].strip()
        return {"action": "keywords", "payload": text_for_keywords}
    else:
        return {"action": "general", "payload": query}

In [22]:
# 🤖 AGENT FUNCTION (IMPLEMENTED)

def agent(query: str):
    try:
        # Route the query using the router_function
        route = router_function(query)
        action = route["action"]
        payload = route["payload"]

        # Define action handlers
        action_handlers = {
            "calculate": lambda p: {
                "type": "calculation",
                "result": calculator(p) if p else "No expression provided to calculate."
            },
            "keywords": lambda p: {
                "type": "keywords",
                "result": extract_keywords(p) if p else "No keywords could be extracted."
            },
            "general": lambda p: {
                "type": "general",
                "result": f"I received your query: \"{p}\". No specific tool matched, so here is a general acknowledgement."
            }
        }

        # Get the handler for the determined action, or a default error handler
        handler = action_handlers.get(action, lambda p: {
            "type": "error",
            "result": "Agent encountered an unknown routing action."
        })

        # Execute the handler and get the result
        result = handler(payload)

        # Handle specific error messages from tools
        if result.get("type") == "calculation" and result.get("result") == "Error in calculation":
            result["type"] = "error"
        elif result.get("type") == "keywords" and not result.get("result"):
            result["type"] = "error"
            result["result"] = "No keywords could be extracted."

        return result

    # --- Basic error handling for anything unexpected ---
    except Exception as e:
        return {"type": "error", "result": f"Agent encountered an unexpected error: {str(e)}"}

## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [23]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['intelligence', 'artificial', 'transforming', 'industries']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': 'I received your query: "What is machine learning?". No specific tool matched, so here is a general acknowledgement.'}
--------------------------------------------------


In [ ]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

Enter query (type 'exit' to stop): extract keywords from Machine learning models require large datasets for training
Response: {'type': 'keywords', 'result': ['large', 'models', 'require', 'datasets', 'learning']}
Enter query (type 'exit' to stop): Hello, how are you?
Response: {'type': 'general', 'result': 'I received your query: "Hello, how are you?". No specific tool matched, so here is a general acknowledgement.'}
Enter query (type 'exit' to stop): calculate two plus two
Response: {'type': 'calculation', 'result': 'Invalid mathematical expression: invalid syntax (<string>, line 1)'}


## Sample Test Queries

### Calculator Route
- calculate 15 * 4
- calculate 100 / 5
- calculate (12 + 8) * 2
- calculate 50 - 7

### Keyword Extraction Route
- extract keywords from Machine learning models require large datasets for training
- find keywords in Climate change is affecting global weather patterns
- keywords please: Python programming is widely used in data science

### General Fallback Route
- Hello, how are you?
- What is the capital of France?
- Tell me a joke

### Edge Cases
- calculate → No expression provided to calculate
- calculate two plus two → Error in calculation (not valid Python syntax)
- keywords → returns ['keywords'] itself, since it's 8 letters long

**Note:** Don't wrap input in quotes — type `calculate 20 + 5`, not `"calculate 20 + 5"`.